# Expectiminimax Algorithm

Expectiminimax is a variation of the Minimax algorithm, used in games with **elements of chance** (e.g., Backgammon, Monopoly, or games with probabilistic opponent behaviors).

### Node Types
1. **MAX Nodes:** The player tries to maximize the score.
2. **MIN Nodes:** The opponent tries to minimize the score.
3. **CHANCE Nodes:** Represents random events. The value of a chance node is the **expected value** (weighted average) of its children's values.

### Formula for Chance Nodes
$$V(s) = \sum_{s' \in Children(s)} P(s') \times V(s')$$
Where $P(s')$ is the probability of transitioning to state $s'$.


### Step 1: Game Node with Chance Types

We define a node structure that supports three types of nodes:
- `"MAX"`
- `"MIN"`
- `"CHANCE"`

For CHANCE nodes, each child is represented as a tuple of `(probability, child_node)`.
For other nodes, it is just a list of child nodes.


In [2]:
class ExpectiNode:
    def __init__(self, name, node_type="MAX", children=None, value=None):
        self.name = name
        self.node_type = node_type  # "MAX", "MIN", "CHANCE"
        # For CHANCE: list of tuples: (probability, child_node)
        # For MAX/MIN: list of ExpectiNode objects
        self.children = children if children is not None else []
        self.value = value

    def is_terminal(self):
        return self.value is not None


### Step 2: Expectiminimax Implementation

This function recursively evaluates the game tree based on the node type:
- If terminal, returns the static evaluation.
- If MAX, returns the maximum value among children.
- If MIN, returns the minimum value among children.
- If CHANCE, returns the sum of (probability * child_value) for all children.


In [3]:
def expectiminimax(node):
    if node.is_terminal():
        return node.value

    if node.node_type == "MAX":
        best_value = float('-inf')
        for child in node.children:
            best_value = max(best_value, expectiminimax(child))
        return best_value

    elif node.node_type == "MIN":
        best_value = float('inf')
        for child in node.children:
            best_value = min(best_value, expectiminimax(child))
        return best_value

    elif node.node_type == "CHANCE":
        expected_value = 0.0
        for probability, child in node.children:
            expected_value += probability * expectiminimax(child)
        return expected_value


### Step 3: Create a Game Tree with Chance Nodes

We construct a tree where a MAX player makes a decision, which leads to a CHANCE node 
(e.g., throwing a biased coin or rolling a die), followed by terminal leaf values.

Tree structure:
                Root (MAX)
               /          \
       Chance1 (CHANCE)   Chance2 (CHANCE)
       / (0.7)  \ (0.3)   / (0.5)  \ (0.5)
      L1[10]    L2[20]   L3[3]     L4[30]


In [4]:
# Leaf nodes (Terminal states)
l1 = ExpectiNode("L1", value=10)
l2 = ExpectiNode("L2", value=20)
l3 = ExpectiNode("L3", value=3)
l4 = ExpectiNode("L4", value=30)

# Chance nodes with their probabilities
chance1 = ExpectiNode("Chance1", node_type="CHANCE", children=[(0.7, l1), (0.3, l2)])
chance2 = ExpectiNode("Chance2", node_type="CHANCE", children=[(0.5, l3), (0.5, l4)])

# Root node (MAX Player)
root = ExpectiNode("Root", node_type="MAX", children=[chance1, chance2])

result = expectiminimax(root)
print(f"Expectiminimax Value at Root: {result}")



Expectiminimax Value at Root: 16.5


We run the algorithm on the root. 
- Chance1 Value: (0.7 * 10) + (0.3 * 20) = 7.0 + 6.0 = 13.0
- Chance2 Value: (0.5 * 3) + (0.5 * 30) = 1.5 + 15.0 = 16.5
- Root (MAX) Value: max(13.0, 16.5) = 16.5
